# SHRUTI — blind signal exploitation, samples to bits



Give SHRUTI an unlabelled recording and it works out how to *rebuild* the signal,
rebuilds it, compares against your samples — then undoes the interleaving,
corrects the errors and reads the message.

Runs **CPU-only**. No GPU, no accelerator, no cloud service.

> Colab limits worth stating: 12 h session cap, 90 min idle timeout, no
> guaranteed accelerator — none of which matter here.

## 1 · Install

In [ ]:
!git clone -q https://github.com/Parikshat-118/SHRUTI.git
%cd shruti
!pip install -q -e .
import shruti; print('SHRUTI', shruti.__version__)

## 2 · The waveform library

A waveform is **data, not code** — a ~40-line YAML file. Adding one is writing a
text file, which is what makes "we can support a new emitter next month" true
rather than rhetorical.

In [ ]:
from shruti.cartridge import load_library
lib = load_library()
print(lib.summary())
for c in lib:
    print(f'  {c.digest()}  {c.describe()}')

## 3 · The blind self-test

Type any sentence. The transmitter picks a random modulation, symbol rate,
interleaver and code. The analyser is handed an unlabelled file and **told
nothing** — the generator and analyser share no state.

There is no partial-credit version of this: either your sentence comes back or
it does not.

In [ ]:
from shruti.selftest import run_blind_selftest

MESSAGE = 'Type anything you like here and run this cell.'
run_blind_selftest(message=MESSAGE, seed=0)

## 4 · Synthesise a capture, then analyse it blind

The twin is also a **test oracle**: every parameter SHRUTI is asked to recover
is an *input* to the synthesiser, so ground truth is free.

In [ ]:
from shruti.core.bits import text_to_bits
from shruti.l4_twin.chain import synthesise, SynthConfig
from shruti.l0_container import write_sigmf
import numpy as np

cart = lib.by_id('ccsds/tm-concatenated')   # CCSDS telemetry downlink
bits = np.tile(text_to_bits('CCSDS telemetry downlink. '), 60)
r = synthesise(cart, bits, SynthConfig(sps=8, snr_db=12, cfo_hz=250.0, seed=7))
print(r.summary())
meta = write_sigmf('capture', r.samples, r.fs, description='unlabelled')
print('wrote', meta)

In [ ]:
from shruti.pipeline import analyse
res = analyse(str(meta))
print(res.summary())
print()
print('payload:', res.payload_text(120))

## 5 · Spectrum, waterfall and constellation

Spectrum, constellation and waterfall (time–frequency).

In [ ]:
import matplotlib.pyplot as plt
from shruti.web import plots
from shruti.l5_demod import demodulate

sp = plots.spectrum(r.samples, r.fs)
wf = plots.waterfall(r.samples, r.fs)
p  = res.proposals[0]
d  = demodulate(r.samples, p.modulation, p.sps, r.fs)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(np.array(sp['freq_hz'])/1e3, sp['power_db'], lw=0.7)
ax[0].set(title='Spectrum', xlabel='kHz', ylabel='dB')
ax[1].plot(d.symbols.real, d.symbols.imag, '.', ms=1, alpha=0.4)
ax[1].set(title=f'Constellation — {p.modulation.describe()}', aspect='equal')
ax[2].imshow(np.array(wf['rows']), aspect='auto', origin='lower', cmap='viridis')
ax[2].set(title='Waterfall (time-frequency)')
plt.tight_layout(); plt.show()

## 6 · Honest refusal

Capabilities grey themselves out when the capture cannot support them, with the
bit count required. **A tool that refuses is a tool an analyst can use** — and
few tools report their own failures on purpose.

In [ ]:
short = synthesise(cart, text_to_bits('too short'), SynthConfig(sps=8, snr_db=20, seed=1))
m2 = write_sigmf('short', short.samples, short.fs)
r2 = analyse(str(m2))
print('VERDICT:', r2.verdict)
for c in r2.capabilities:
    print(' ', c.describe())

---

Apache-2.0 · every runtime dependency is BSD/MIT/Apache · runs air-gapped.

`github.com/Parikshat-118/SHRUTI`